# Line Model — Byte-Level BPE + RoPE

Encoder-decoder for line completion. Two architectural upgrades:
* **Byte-level BPE tokenizer** (shared with the Token model).
* **RoPE inside *self-attention*** (encoder + decoder). Cross-attention stays standard — Q (decoder positions) and K (encoder positions) live in different position spaces, so rotating both with the same RoPE doesn't make sense.

Custom encoder-decoder blocks because `nn.Transformer` doesn't expose Q/K for rotation. Uses `F.scaled_dot_product_attention` for the efficient masked path (causal, padding, cross). Keeps warmup+cosine, label smoothing, mixed precision, and early stopping.

In [ ]:
%tb
import os, json, math, random, glob, tempfile
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

from tokenizers import Tokenizer
from tokenizers.implementations import ByteLevelBPETokenizer

import warnings
warnings.filterwarnings("ignore")

from modules.Plotting       import MetricLog, plot_metrics
from modules.EarlyStopping  import EarlyStopping
from modules.HandTesting    import hand_test_repl
from modules.BestModelSaver import BestModelSaver

WORKDIR = r'C:\Users\Roman\Documents\Projects\code_autocomplete'
print(f"WORKDIR: {WORKDIR}")
LINE_MODEL_NAME = 'line_model_bpe_rope'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Device] {device}")

## Tokenizer — Byte-Level BPE

Identical to the Token model's tokenizer — train once, reuse the saved `tokenizer_bpe.json` between both notebooks.

In [ ]:
SPECIAL = {"<PAD>": 0, "<UNK>": 1, "<BOS>": 2, "<EOS>": 3}


class _IdToTokenView:
    def __init__(self, tk): self.tk = tk
    def get(self, i, default=""):
        t = self.tk.id_to_token(i); return t if t is not None else default
    def __getitem__(self, i):
        t = self.tk.id_to_token(i)
        if t is None: raise KeyError(i)
        return t
    def __contains__(self, i): return self.tk.id_to_token(i) is not None


class BPECodeTokenizer:
    SPECIAL_TOKENS = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]

    def __init__(self, vocab_size=16000, min_freq=2):
        self.vocab_size = vocab_size
        self.min_freq   = min_freq
        self.tk         = None

    def build(self, texts):
        bpe = ByteLevelBPETokenizer()
        bpe.train_from_iterator(
            iter(texts),
            vocab_size=self.vocab_size,
            min_frequency=self.min_freq,
            special_tokens=self.SPECIAL_TOKENS,
        )
        with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
            tmp = f.name
        bpe.save(tmp)
        self.tk = Tokenizer.from_file(tmp)
        os.unlink(tmp)
        print(f"[BPE Tokenizer] vocab_size={self.tk.get_vocab_size()}")

    def encode(self, text):
        return [self.bos_id] + self.tk.encode(text).ids + [self.eos_id]

    def decode(self, ids):
        clean = [i for i in ids if i not in (self.pad_id, self.bos_id, self.eos_id)]
        return self.tk.decode(clean)

    def save(self, path):
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        self.tk.save(str(path))

    @classmethod
    def load(cls, path):
        obj = cls()
        obj.tk = Tokenizer.from_file(str(path))
        obj.vocab_size = obj.tk.get_vocab_size()
        return obj

    @property
    def vocab(self):  return self.tk.get_vocab_size()
    @property
    def pad_id(self): return self.tk.token_to_id("<PAD>") or 0
    @property
    def unk_id(self): return self.tk.token_to_id("<UNK>") or 1
    @property
    def bos_id(self): return self.tk.token_to_id("<BOS>") or 2
    @property
    def eos_id(self): return self.tk.token_to_id("<EOS>") or 3
    @property
    def id2token(self): return _IdToTokenView(self.tk)
    @property
    def built(self):  return self.tk is not None

## Datasets

In [ ]:
def load_files(data_dir, max_files=0):
    patterns = ["**/*.py", "**/*.txt"]
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(data_dir, pat), recursive=True))
    if max_files: files = files[:max_files]
    texts = []
    for fp in files:
        try: texts.append(Path(fp).read_text(errors="replace"))
        except Exception: pass
    print(f"[Data] loaded {len(texts)} files from {data_dir}")
    return texts


class LineDataset(Dataset):
    """One sample = (prefix_tokens, full_line_tokens)."""
    def __init__(self, texts, tokenizer, max_prefix=96, max_line=64):
        self.samples = []
        for text in texts:
            for line in text.splitlines():
                # strip inline comments (but not inside strings)
                cp = line.find('#')
                if cp != -1 and cp > 0 and line[cp - 1] not in ('\'', '"'):
                    line = line[:cp]
                line = line.rstrip()
                if len(line.strip()) < 10:
                    continue
                full = tokenizer.encode(line)
                if len(full) < 4:
                    continue
                # split at 30-70% so the prefix is informative
                lo = max(2, int(len(full) * 0.3))
                hi = max(lo + 1, int(len(full) * 0.7))
                split = random.randint(lo, hi)
                prefix = full[:split][-max_prefix:]
                target = full[split:][:max_line]
                target.append(tokenizer.eos_id)
                self.samples.append((prefix, target))
        print(f"[LineDataset] {len(self.samples)} samples")

    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


def collate_line(batch, pad_id):
    prefixes, targets = zip(*batch)
    max_p = max(len(p) for p in prefixes)
    max_t = max(len(t) for t in targets)
    P = torch.full((len(batch), max_p), pad_id, dtype=torch.long)
    T = torch.full((len(batch), max_t), pad_id, dtype=torch.long)
    for i, (p, t) in enumerate(zip(prefixes, targets)):
        P[i, :len(p)] = torch.tensor(p)
        T[i, :len(t)] = torch.tensor(t)
    return P, T

## RoPE primitives

In [ ]:
class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=4096, base=10000):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        self.head_dim = head_dim
        self._cached_len = 0
        self._build_cache(max_seq_len, torch.device("cpu"))
    def _build_cache(self, seq_len, device):
        t     = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq.to(device))
        emb   = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos(), persistent=False)
        self.register_buffer("sin_cached", emb.sin(), persistent=False)
        self._cached_len = seq_len
    def forward(self, seq_len, device):
        if seq_len > self._cached_len or self.cos_cached.device != device:
            self._build_cache(max(seq_len, self._cached_len * 2), device)
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, cos, sin):
    cos = cos.unsqueeze(0).unsqueeze(0); sin = sin.unsqueeze(0).unsqueeze(0)
    return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

## Encoder-Decoder model with RoPE

* `BiSelfAttentionRoPE` — bidirectional self-attention with RoPE (encoder).
* `CausalSelfAttentionRoPE` — causal self-attention with RoPE + optional padding mask (decoder).
* `CrossAttention` — standard MHA, no RoPE (Q-decoder vs K/V-encoder).
* `EncoderBlockRoPE`, `DecoderBlockRoPE` — pre-norm sub-blocks with residual connections.

In [ ]:
@dataclass
class ModelCfg:
    vocab:     int   = 16000
    d_model:   int   = 256
    n_heads:   int   = 8
    n_layers:  int   = 4
    d_ff:      int   = 1024
    max_len:   int   = 256
    dropout:   float = 0.1
    rope_base: int   = 10000


class BiSelfAttentionRoPE(nn.Module):
    """Bidirectional self-attention with RoPE — used in the encoder."""
    def __init__(self, d_model, n_heads, dropout, rope):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv  = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = dropout
        self.resid_drop = nn.Dropout(dropout)
        self.rope = rope

    def forward(self, x, key_padding_mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)                       # (B, H, T, hd) each
        cos, sin = self.rope(T, x.device); cos = cos.to(q.dtype); sin = sin.to(q.dtype)
        q, k = apply_rope(q, k, cos, sin)

        attn_mask = None
        if key_padding_mask is not None:
            attn_mask = key_padding_mask[:, None, None, :]         # (B, 1, 1, T) bool, True = pad
        out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_mask,
            dropout_p=self.attn_drop if self.training else 0.0,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(out))


class CausalSelfAttentionRoPE(nn.Module):
    """Causal self-attention with RoPE — used in the decoder."""
    def __init__(self, d_model, n_heads, dropout, rope):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv  = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = dropout
        self.resid_drop = nn.Dropout(dropout)
        self.rope = rope

    def forward(self, x, key_padding_mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        cos, sin = self.rope(T, x.device); cos = cos.to(q.dtype); sin = sin.to(q.dtype)
        q, k = apply_rope(q, k, cos, sin)

        if key_padding_mask is not None:
            # combine causal mask with padding mask → (B, 1, T, T) bool
            causal = torch.triu(
                torch.ones(T, T, dtype=torch.bool, device=x.device), diagonal=1
            )                                                       # (T, T)
            pad = key_padding_mask[:, None, None, :]                # (B, 1, 1, T)
            attn_mask = causal[None, None, :, :] | pad              # (B, 1, T, T)
            out = F.scaled_dot_product_attention(
                q, k, v, attn_mask=attn_mask,
                dropout_p=self.attn_drop if self.training else 0.0,
            )
        else:
            out = F.scaled_dot_product_attention(
                q, k, v, is_causal=True,
                dropout_p=self.attn_drop if self.training else 0.0,
            )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(out))


class CrossAttention(nn.Module):
    """Standard cross-attention (no RoPE) — Q from decoder, K/V from encoder."""
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.q_proj  = nn.Linear(d_model, d_model, bias=False)
        self.kv_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.proj    = nn.Linear(d_model, d_model, bias=False)
        self.attn_drop = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, memory, memory_padding_mask=None):
        B, T, C = x.shape
        _, S, _ = memory.shape
        q  = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        kv = self.kv_proj(memory).view(B, S, 2, self.n_heads, self.head_dim)
        k, v = kv.permute(2, 0, 3, 1, 4)
        attn_mask = None
        if memory_padding_mask is not None:
            attn_mask = memory_padding_mask[:, None, None, :]
        out = F.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_mask,
            dropout_p=self.attn_drop if self.training else 0.0,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(out))


class EncoderBlockRoPE(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout, rope):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = BiSelfAttentionRoPE(d_model, n_heads, dropout, rope)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )
    def forward(self, x, key_padding_mask=None):
        x = x + self.attn(self.norm1(x), key_padding_mask=key_padding_mask)
        x = x + self.ffn (self.norm2(x))
        return x


class DecoderBlockRoPE(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout, rope):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attn = CausalSelfAttentionRoPE(d_model, n_heads, dropout, rope)
        self.norm2 = nn.LayerNorm(d_model)
        self.cross_attn = CrossAttention(d_model, n_heads, dropout)
        self.norm3 = nn.LayerNorm(d_model)
        self.ffn   = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout),
        )
    def forward(self, x, memory, tgt_padding_mask=None, memory_padding_mask=None):
        x = x + self.self_attn(self.norm1(x), key_padding_mask=tgt_padding_mask)
        x = x + self.cross_attn(self.norm2(x), memory, memory_padding_mask=memory_padding_mask)
        x = x + self.ffn(self.norm3(x))
        return x


class LineModel(nn.Module):
    """Encoder-Decoder Transformer with RoPE in self-attention."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg     = cfg
        self.enc_emb = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
        self.dec_emb = nn.Embedding(cfg.vocab, cfg.d_model, padding_idx=0)
        head_dim     = cfg.d_model // cfg.n_heads
        # one RoPE shared across both encoder + decoder self-attention
        self.rope    = RotaryEmbedding(head_dim, max_seq_len=max(cfg.max_len, 1024), base=cfg.rope_base)

        self.enc_blocks = nn.ModuleList([
            EncoderBlockRoPE(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout, self.rope)
            for _ in range(cfg.n_layers)
        ])
        self.dec_blocks = nn.ModuleList([
            DecoderBlockRoPE(cfg.d_model, cfg.n_heads, cfg.d_ff, cfg.dropout, self.rope)
            for _ in range(cfg.n_layers)
        ])
        self.enc_norm = nn.LayerNorm(cfg.d_model)
        self.dec_norm = nn.LayerNorm(cfg.d_model)
        self.head     = nn.Linear(cfg.d_model, cfg.vocab, bias=False)
        self.dec_emb.weight = self.head.weight                     # weight tying
        self.drop = nn.Dropout(cfg.dropout)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def encode(self, src, src_key_padding_mask=None):
        h = self.drop(self.enc_emb(src))
        for block in self.enc_blocks:
            h = block(h, key_padding_mask=src_key_padding_mask)
        return self.enc_norm(h)

    def forward(self, src, tgt,
                src_key_padding_mask=None,
                tgt_key_padding_mask=None):
        memory = self.encode(src, src_key_padding_mask=src_key_padding_mask)
        h = self.drop(self.dec_emb(tgt))
        for block in self.dec_blocks:
            h = block(h, memory,
                      tgt_padding_mask=tgt_key_padding_mask,
                      memory_padding_mask=src_key_padding_mask)
        return self.head(self.dec_norm(h))

    @torch.no_grad()
    def generate(self, prefix_ids, max_new=64, temperature=0.7, top_k=40, tokenizer=None):
        self.eval()
        dev = next(self.parameters()).device
        src = torch.tensor([prefix_ids], dtype=torch.long, device=dev)
        src_pad = (src == 0)
        memory  = self.encode(src, src_key_padding_mask=src_pad)

        bos = tokenizer.bos_id if tokenizer is not None else SPECIAL["<BOS>"]
        eos = tokenizer.eos_id if tokenizer is not None else SPECIAL["<EOS>"]
        dec_ids = [bos]; out_ids = []
        for _ in range(max_new):
            tgt = torch.tensor([dec_ids], dtype=torch.long, device=dev)
            h = self.drop(self.dec_emb(tgt))
            for block in self.dec_blocks:
                h = block(h, memory, memory_padding_mask=src_pad)
            logits = self.head(self.dec_norm(h))[0, -1] / max(temperature, 1e-6)
            if top_k:
                topk_v, _ = torch.topk(logits, top_k)
                logits[logits < topk_v[-1]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, 1).item()
            if nxt == eos: break
            dec_ids.append(nxt); out_ids.append(nxt)
        return out_ids

## Training loop

In [ ]:
def _clip_norm(model, max_norm=1.0):
    return nn.utils.clip_grad_norm_(model.parameters(), max_norm).item()


def train_line_model(model, train_dl, val_dl, epochs, lr, device,
                     saver, log, plot_dir,
                     label_smoothing=0.1, warmup_frac=0.05,
                     patience=3, use_amp=True):
    tqdm.write(f"[Line] DataLoader — {len(train_dl)} train batches, {len(val_dl)} val batches")
    opt          = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    total_steps  = len(train_dl) * epochs
    warmup_steps = max(1, int(total_steps * warmup_frac))
    sched        = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)
    crit         = nn.CrossEntropyLoss(ignore_index=SPECIAL["<PAD>"], label_smoothing=label_smoothing)
    amp_enabled  = use_amp and device.type == "cuda"
    scaler       = GradScaler("cuda", enabled=amp_enabled)
    stopper      = EarlyStopping(patience=patience)
    PAD = SPECIAL["<PAD>"]

    for ep in range(1, epochs + 1):
        print(f"Epoch {ep}")
        model.train()
        t_loss = t_acc = t_steps = 0; gn = 0.0
        for src, tgt in tqdm(train_dl, desc=f"[Line] Epoch {ep}/{epochs} train",
                             leave=False, unit="batch"):
            src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
            src_pad  = (src == PAD)
            dec_in   = tgt[:, :-1]
            dec_out  = tgt[:,  1:]
            tgt_pad  = (dec_in == PAD)

            opt.zero_grad(set_to_none=True)
            with autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                logits = model(src, dec_in,
                               src_key_padding_mask=src_pad,
                               tgt_key_padding_mask=tgt_pad)
                loss   = crit(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))

            scaler.scale(loss).backward(); scaler.unscale_(opt)
            gn = _clip_norm(model)
            scaler.step(opt); scaler.update(); sched.step()

            with torch.no_grad():
                preds = logits.argmax(-1)
                mask  = (dec_out != PAD)
                t_acc += (preds[mask] == dec_out[mask]).float().mean().item()
            t_loss  += loss.item()
            t_steps += 1

        tl, ta = t_loss / t_steps, t_acc / t_steps

        model.eval()
        v_loss = v_steps = 0
        with torch.no_grad():
            for src, tgt in tqdm(val_dl, desc=f"[Line] Epoch {ep}/{epochs} val  ",
                                 leave=False, unit="batch"):
                src, tgt = src.to(device, non_blocking=True), tgt.to(device, non_blocking=True)
                src_pad  = (src == PAD)
                dec_in   = tgt[:, :-1]; dec_out = tgt[:, 1:]
                tgt_pad  = (dec_in == PAD)
                with autocast("cuda", dtype=torch.float16, enabled=amp_enabled):
                    logits = model(src, dec_in,
                                   src_key_padding_mask=src_pad,
                                   tgt_key_padding_mask=tgt_pad)
                    loss   = crit(logits.reshape(-1, logits.size(-1)), dec_out.reshape(-1))
                v_loss  += loss.item()
                v_steps += 1
        vl = v_loss / v_steps if v_steps else tl

        log.append(train_loss=tl, val_loss=vl,
                   train_ppl=math.exp(min(tl, 20)), val_ppl=math.exp(min(vl, 20)),
                   lr=opt.param_groups[0]["lr"], token_acc=ta, grad_norm=gn)
        tqdm.write(f"[Line  ep {ep:3d}] train_loss={tl:.4f}  val_loss={vl:.4f}"
                   f"  ppl={math.exp(min(vl,20)):.1f}  acc={ta:.3f}"
                   f"  lr={opt.param_groups[0]['lr']:.2e}")
        saver.save(model, vl, ep)
        if ep % max(1, epochs // 5) == 0 or ep == epochs:
            plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Epoch {ep}",
                         f"{plot_dir}/{LINE_MODEL_NAME}_ep{ep:02d}.png")
        if stopper(vl):
            tqdm.write(f"[Early stop] val_loss did not improve for {stopper.patience} epochs — stopping.")
            break
    plot_metrics(log, f"{LINE_MODEL_NAME.replace('_', ' ')} — Final",
                 f"{plot_dir}/{LINE_MODEL_NAME}_final.png")

## Main

In [ ]:
class Arguments():
    def __init__(self,
                 data_dir=f"{WORKDIR}/Clean_Dataset",
                 ckpt_dir=f"{WORKDIR}/checkpoints/{LINE_MODEL_NAME}",
                 plot_dir=f"{WORKDIR}/plots/{LINE_MODEL_NAME}",
                 tokenizer=f"{WORKDIR}/tokenizer_bpe.json",
                 epochs=5, batch=32, lr=5e-4, ctx=128,
                 d_model=256, n_layers=4, n_heads=8,
                 vocab_size=16000, max_files=0, val_split=0.1, seed=42,
                 label_smoothing=0.1, warmup_frac=0.05, patience=3, use_amp=True,
                 skip_line=False, test=False):
        for k, v in locals().items():
            if k != "self": setattr(self, k, v)


def main():
    args = Arguments()
    # args = Arguments(max_files=200, epochs=2)
    # args = Arguments(test=True)

    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(args.seed)
    os.makedirs(args.ckpt_dir, exist_ok=True)
    os.makedirs(args.plot_dir, exist_ok=True)

    # ── BPE tokenizer (re-use the file the Token model trained, if it exists) ─
    if os.path.exists(args.tokenizer):
        print(f"[Tokenizer] loading {args.tokenizer}")
        tokenizer = BPECodeTokenizer.load(args.tokenizer)
    else:
        print("[Tokenizer] training byte-level BPE from data …")
        texts = load_files(args.data_dir, args.max_files)
        if not texts: print("[ERROR] no data files found."); return
        tokenizer = BPECodeTokenizer(vocab_size=args.vocab_size)
        tokenizer.build(texts)
        tokenizer.save(args.tokenizer)
        print(f"[Tokenizer] saved to {args.tokenizer}")

    cfg = ModelCfg(
        vocab=tokenizer.vocab, d_model=args.d_model,
        n_heads=args.n_heads, n_layers=args.n_layers,
        d_ff=args.d_model * 4, max_len=args.ctx + 32,
    )
    torch.serialization.add_safe_globals([ModelCfg])

    if args.test:
        lm = LineModel(cfg).to(device)
        line_paths = sorted(glob.glob(str(Path(args.ckpt_dir) / f"{LINE_MODEL_NAME}_*.pt")))
        if line_paths:
            ck = torch.load(line_paths[0], map_location=device, weights_only=False)
            lm.load_state_dict(ck["model_state"])
            print(f"[Loaded] line model from {line_paths[0]}")
        hand_test_repl(None, lm, tokenizer, None, device); return

    print("[Loading] Started loading")
    texts = load_files(args.data_dir, args.max_files)
    if not texts: print("[ERROR] no data files found."); return
    print("[Loading] Ended loading")

    random.shuffle(texts)
    split = max(1, int(len(texts) * (1 - args.val_split)))
    tr_txt, va_txt = texts[:split], texts[split:]

    if not args.skip_line:
        print("  Preparing LINE model (BPE + RoPE)")
        tr_ds = LineDataset(tr_txt, tokenizer)
        va_ds = LineDataset(va_txt, tokenizer)
        collate = lambda b: collate_line(b, tokenizer.pad_id)
        tr_dl = DataLoader(tr_ds, args.batch, shuffle=True,
                           collate_fn=collate, num_workers=0, pin_memory=True)
        va_dl = DataLoader(va_ds, args.batch, shuffle=False,
                           collate_fn=collate, num_workers=0, pin_memory=True)

        line_model = LineModel(cfg).to(device)
        n_params   = sum(p.numel() for p in line_model.parameters() if p.requires_grad)
        print(f"[Line  Model] {n_params/1e6:.2f}M parameters (BPE+RoPE)")

        line_saver = BestModelSaver(args.ckpt_dir, LINE_MODEL_NAME)
        line_log   = MetricLog()
        print("  Training LINE model")
        train_line_model(
            model=line_model, train_dl=tr_dl, val_dl=va_dl,
            epochs=args.epochs, lr=args.lr, device=device,
            saver=line_saver, log=line_log, plot_dir=args.plot_dir,
            label_smoothing=args.label_smoothing, warmup_frac=args.warmup_frac,
            patience=args.patience, use_amp=args.use_amp,
        )
        hand_test_repl(None, line_model, tokenizer, None, device)


main()